In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import svds

# Загружаем датасет

In [2]:
data = pd.read_csv('/kaggle/input/user-item-data-recsys/ratings.csv', 
                   usecols=['userId', 'movieId', 'rating', 'timestamp'])

data.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [3]:
# смотрим на размерность
num_users, num_movies = data.userId.nunique(), data.movieId.nunique()
num_users, num_movies, data.shape[0]

(200948, 84432, 32000204)

# Фильтруем

In [4]:
train_df = data.loc[(data.timestamp < 1514764800)].copy()
test_df = data.loc[(data.timestamp >= 1514764800)].copy()

def fast_iterative_filtering(train_df, test_df, max_iterations=10):
    """
    Быстрая итеративная фильтрация
    Оставляет только пересечение пользователей и фильмов в ОБОИХ наборах
    """
    print("Начало...")
    
    for iteration in range(max_iterations):
        print(f"\nИтерация {iteration + 1}:")
        
        # Сохраняем предыдущие размеры для проверки сходимости
        prev_train_len = len(train_df)
        prev_test_len = len(test_df)
        
        # 1. НАХОДИМ ОБЩИХ ПОЛЬЗОВАТЕЛЕЙ
        train_users = train_df['userId'].unique()
        test_users = test_df['userId'].unique()
        common_users = np.intersect1d(train_users, test_users)
        
        print(f"    Общих пользователей: {len(common_users)}")
        
        # 2. ФИЛЬТРУЕМ ОБА НАБОРА по общим пользователям
        train_df = train_df[train_df['userId'].isin(common_users)].copy()
        test_df = test_df[test_df['userId'].isin(common_users)].copy()
        
        # 3. НАХОДИМ ОБЩИЕ ФИЛЬМЫ
        train_movies = train_df['movieId'].unique()
        test_movies = test_df['movieId'].unique()
        common_movies = np.intersect1d(train_movies, test_movies)
        
        print(f"    Общих фильмов: {len(common_movies)}")
        
        # 4. ФИЛЬТРУЕМ ОБА НАБОРА по общим фильмам
        train_df = train_df[train_df['movieId'].isin(common_movies)].copy()
        test_df = test_df[test_df['movieId'].isin(common_movies)].copy()
        
        # 5. ПРОВЕРКА СХОДИМОСТИ
        current_train_len = len(train_df)
        current_test_len = len(test_df)
        
        print(f"    Записей в трейне: {current_train_len:,} (было {prev_train_len:,})")
        print(f"    Записей в тесте:  {current_test_len:,} (было {prev_test_len:,})")
        
        # Если размеры перестали меняться - выходим
        if (current_train_len == prev_train_len and 
            current_test_len == prev_test_len):
            print(f"\n>>>>> Стабилизация достигнута за {iteration + 1} итераций<<<<<")
            break
            
        if iteration == max_iterations - 1:
            print(f"\n!!!1Достигнут максимум {max_iterations} итераций!!!")
    
    return train_df, test_df

In [5]:
train_df, test_df = fast_iterative_filtering(train_df, test_df, max_iterations=10)

Начало...

Итерация 1:
    Общих пользователей: 8567
    Общих фильмов: 33882
    Записей в трейне: 3,648,210 (было 24,588,450)
    Записей в тесте:  884,868 (было 7,411,754)

Итерация 2:
    Общих пользователей: 8482
    Общих фильмов: 33869
    Записей в трейне: 3,618,924 (было 3,648,210)
    Записей в тесте:  884,832 (было 884,868)

Итерация 3:
    Общих пользователей: 8482
    Общих фильмов: 33869
    Записей в трейне: 3,618,924 (было 3,618,924)
    Записей в тесте:  884,832 (было 884,832)

>>>>> Стабилизация достигнута за 3 итераций<<<<<


In [6]:
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import csr_matrix

# 1. Кодируем пользователей и фильмы
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

train_df['userId_enc'] = user_encoder.fit_transform(train_df['userId'])
train_df['movieId_enc'] = item_encoder.fit_transform(train_df['movieId'])

test_df['userId_enc'] = user_encoder.transform(test_df['userId'])
test_df['movieId_enc'] = item_encoder.transform(test_df['movieId'])

print(train_df['userId_enc'].nunique(), train_df['movieId_enc'].nunique())
print(test_df['userId_enc'].nunique(), test_df['movieId_enc'].nunique())

8482 33869
8482 33869


# Задаем метрики

In [7]:
import numpy as np

def dcg(scores):
    return np.sum((2**scores-1) / np.log2(np.arange(len(scores)) + 2))

def ndcg_metric(gt_items, predicted, k=15):
    predicted = predicted[:k]
    relevance = np.array([1 if x in gt_items else 0 for x in predicted]) # 1 - смотрел, 0 - не смотрел.
    rank_dcg = dcg(relevance)
    ideal_dcg = dcg(np.sort(relevance)[::-1]) # [0,0,0,0,1,1] -> [1,1,0,0,0,0] - идеальное ранжирование  
    return rank_dcg / ideal_dcg if ideal_dcg > 0 else 0

def precision_at_k(gt_items, predicted, k=15):
    predicted = predicted[:k]
    return len(set(predicted) & set(gt_items)) / k

def recall_at_k(gt_items, predicted, k=15):
    if len(gt_items) == 0:
        return 0.0
    predicted = predicted[:k]
    return len(set(predicted) & set(gt_items)) / len(gt_items)

def evaluate_recommendations(test_df, recommendations, k_ndcg=15, k_prec=15, k_rec=50):
    ndcg_list, prec_list, rec_list = [], [], []
    
    for u, group in test_df.groupby('userId_enc'):
        if u not in recommendations:  # Проверка!
            continue
        
        true_items = set(group['movieId_enc'])
        if len(true_items) == 0:  # Проверка!
            continue
        
        recs = recommendations[u]
        # проверка знаний - сравнеине с тем, что подсказал помощник и с true_items
        
        ndcg_list.append(ndcg_metric(true_items, recs, k=k_ndcg))
        prec_list.append(precision_at_k(true_items, recs, k=k_prec))
        rec_list.append(recall_at_k(true_items, recs, k=k_rec))
    
    if not ndcg_list:  # Пустой список
        return 0.0, 0.0, 0.0
    # после проверки всех студентов возвращаем одно среднее значение
    return np.mean(ndcg_list), np.mean(prec_list), np.mean(rec_list)

# Baselines: MostPop and RandomRecs

In [8]:
# Получаем список популярных фильмов (по количеству оценок)
most_popular_items = train_df.groupby('movieId_enc')['rating'].count().sort_values(ascending=False).index.tolist()
print(len(most_popular_items))
# Все фильмы (для Random)
all_items = train_df['movieId_enc'].unique()

33869


In [9]:
# MOSTPOP
# создается словарь популярных айтемов 
mostpop_recs = {u: most_popular_items[:50] for u in test_df['userId_enc'].unique()}  # топ-50 для NDCG
ndcg, prec, rec = evaluate_recommendations(test_df, mostpop_recs, k_ndcg=15, k_prec=15, k_rec=50)
print('MOSTPOP - NDCG@15: {:.4f}, P@15: {:.4f}, R@50: {:.4f}'.format(ndcg, prec, rec))

# Random
np.random.seed(42)
random_recs = {u: list(np.random.choice(all_items, 50, replace=False)) for u in test_df['userId_enc'].unique()} # выбираем 50 случайных фильмов без повторений
ndcg, prec, rec = evaluate_recommendations(test_df, random_recs, k_ndcg=15, k_prec=15, k_rec=50)
print('Random - NDCG@15: {:.4f}, P@15: {:.4f}, R@50: {:.4f}'.format(ndcg, prec, rec))

MOSTPOP - NDCG@15: 0.2103, P@15: 0.0744, R@50: 0.0717
Random - NDCG@15: 0.0163, P@15: 0.0031, R@50: 0.0017


In [21]:
# MostPop с разными K и разными метриками
print("MostPop")
for n in [10, 30, 50, 100]:
    mostpop_recs = {u: most_popular_items[:n] for u in test_df['userId_enc'].unique()}
    
    print(f"N={n}:")
    
    # Для N=10 можно считать только @10, для N=30 — @10 и @15 и т.д.
    if n >= 10:
        ndcg10, prec10, rec10 = evaluate_recommendations(test_df, mostpop_recs, 
                                                         k_ndcg=10, k_prec=10, k_rec=min(10, n))
        print(f"  @10: NDCG={ndcg10:.4f}, P={prec10:.4f}, R={rec10:.4f}")
    
    if n >= 15:
        ndcg15, prec15, rec15 = evaluate_recommendations(test_df, mostpop_recs,
                                                         k_ndcg=15, k_prec=15, k_rec=min(15, n))
        print(f"  @15: NDCG={ndcg15:.4f}, P={prec15:.4f}, R={rec15:.4f}")
    
    if n >= 50:
        ndcg50, prec50, rec50 = evaluate_recommendations(test_df, mostpop_recs,
                                                       k_ndcg=50, k_prec=10, k_rec=50)
        print(f"  @50: NDCG={ndcg50:.4f}, P={prec50:.4f}, R={rec50:.4f}")
        
# Random тоже с разными @K
print("\nRandom")
for seed in [42, 123]:
    np.random.seed(seed)
    random_recs = {u: list(np.random.choice(all_items, 50, replace=False)) 
                   for u in test_df['userId_enc'].unique()}
    
    print(f"Seed={seed}:")
    
    # @10
    ndcg10, prec10, rec10 = evaluate_recommendations(test_df, random_recs,
                                                     k_ndcg=10, k_prec=10, k_rec=10)
    print(f"  @10: NDCG={ndcg10:.4f}, P={prec10:.4f}, R={rec10:.4f}")
    
    # @15  
    ndcg15, prec15, rec15 = evaluate_recommendations(test_df, random_recs,
                                                     k_ndcg=15, k_prec=15, k_rec=15)
    print(f"  @15: NDCG={ndcg15:.4f}, P={prec15:.4f}, R={rec15:.4f}")
    
    # @50 (по всем рекомендациям)
    ndcg50, prec50, rec50 = evaluate_recommendations(test_df, random_recs,
                                                     k_ndcg=50, k_prec=10, k_rec=50)
    print(f"  @50: NDCG={ndcg50:.4f}, P={prec50:.4f}, R={rec50:.4f}")

MostPop
N=10:
  @10: NDCG=0.1931, P=0.0772, R=0.0168
N=30:
  @10: NDCG=0.1931, P=0.0772, R=0.0168
  @15: NDCG=0.2103, P=0.0744, R=0.0228
N=50:
  @10: NDCG=0.1931, P=0.0772, R=0.0168
  @15: NDCG=0.2103, P=0.0744, R=0.0228
  @50: NDCG=0.2713, P=0.0772, R=0.0717
N=100:
  @10: NDCG=0.1931, P=0.0772, R=0.0168
  @15: NDCG=0.2103, P=0.0744, R=0.0228
  @50: NDCG=0.2713, P=0.0772, R=0.0717

Random
Seed=42:
  @10: NDCG=0.0126, P=0.0030, R=0.0004
  @15: NDCG=0.0163, P=0.0031, R=0.0006
  @50: NDCG=0.0318, P=0.0030, R=0.0017
Seed=123:
  @10: NDCG=0.0149, P=0.0034, R=0.0004
  @15: NDCG=0.0183, P=0.0034, R=0.0005
  @50: NDCG=0.0343, P=0.0034, R=0.0016


# Более сложные модели
## Создание разряженных матриц

In [10]:
# Создаем sparse-матрицу для train

# 1) Узнаем размеры матрицы
num_users = train_df['userId_enc'].nunique()
num_items = train_df['movieId_enc'].nunique()

# Бинарная для KNN
R_binary = csr_matrix(
    (np.ones(len(train_df)),
     (train_df['userId_enc'], train_df['movieId_enc'])),
    shape=(num_users, num_items)
)

# С оценками для SVD
R_ratings = csr_matrix(
    (train_df['rating'],
     (train_df['userId_enc'], train_df['movieId_enc'])),
    shape=(num_users, num_items)
)

## ItemKNN | UserKNN

In [11]:
item_sim = cosine_similarity(R_binary.T, dense_output=False)

def item_knn_recommendations_fast(R_train, item_sim, top_k=50, top_neighbors=10, batch_size=500):

    n_users, n_items = R_train.shape
    recommendations = {}
    
    # 1. Обрезаем матрицу схожестей до top_neighbors через LIL
    item_sim_lil = lil_matrix(item_sim.shape)
    for i in range(n_items):
        row = item_sim[i].toarray().flatten()
        row[i] = -np.inf  # исключаем сам айтем
        top_indices = np.argsort(row)[::-1][:top_neighbors]
        item_sim_lil[i, top_indices] = item_sim[i, top_indices]
    
    item_sim_top = item_sim_lil.tocsr()  # → CSR для быстрого умножения
    
    # 2. Обработка батчами
    for start in range(0, n_users, batch_size):
        end = min(start + batch_size, n_users)
        
        # Матрица взаимодействий для батча
        user_batch = R_train[start:end, :]
        
        # Умножение: (batch × items) × (items × items) = (batch × items)
        scores_batch = user_batch.dot(item_sim_top)
        scores_batch = scores_batch.toarray() if hasattr(scores_batch, 'toarray') else scores_batch
        
        # user_batch[i, j] = смотрел ли пользователь i фильм j
        # item_sim_top[j, k] = схожесть фильма j с фильмом k (только top_neighbors)
        # Результат scores_batch[i, k] = Сумма по j user_batch[i,j] × item_sim_top[j,k]
        # = сумма схожестей всех просмотренных пользователем фильмов с фильмом k
        
        # Исключаем уже просмотренные
        rated = user_batch.toarray() > 0
        scores_batch[rated] = -np.inf
        
        # Топ-k рекомендаций
        top_indices = np.argsort(scores_batch, axis=1)[:, ::-1][:, :top_k]
        
        # Сохраняем рекомендации
        for i, recs in enumerate(top_indices):
            recommendations[start + i] = recs.tolist()
    
    return recommendations


# Пример использования:

# 2. Генерация рекомендаций
itemknn_recs = item_knn_recommendations_fast(
    R_binary, 
    item_sim, 
    top_k=50, 
    top_neighbors=10, 
    batch_size=500
)

# 3. Оценка
ndcg, prec, rec = evaluate_recommendations(test_df, itemknn_recs, k_ndcg=15, k_prec=15, k_rec=50)
print('Item-KNN - NDCG@15: {:.4f}, P@15: {:.4f}, R@50: {:.4f}'.format(ndcg, prec, rec))

Item-KNN - NDCG@15: 0.4351, P@15: 0.1953, R@50: 0.1198


In [22]:
print("=== Item-KNN GridSearch ===")

# Параметры для перебора
neighbors_list = [5, 10, 20, 50]  # сколько соседей учитывать
top_k_list = [50, 75, 100]        # сколько рекомендовать

# Предвычисляем item_sim один раз (экономия времени)
item_sim = cosine_similarity(R_binary.T, dense_output=False)

for neighbors in neighbors_list:
    for top_k in top_k_list:
        # Генерация рекомендаций
        itemknn_recs = item_knn_recommendations_fast(
            R_binary, 
            item_sim, 
            top_k=top_k, 
            top_neighbors=neighbors, 
            batch_size=500
        )
        
        # Оценка по разным K
        print(f"\nNeighbors={neighbors}, TopK={top_k}:")
        
        # @10, @15, @50
        for k_ndcg, k_prec, k_rec in [(10,10,10), (15,15,15), (50,50,50)]:
            ndcg, prec, rec = evaluate_recommendations(
                test_df, itemknn_recs,
                k_ndcg=k_ndcg, 
                k_prec=k_prec, 
                k_rec=k_rec
            )
            print(f"  @{k_ndcg}: NDCG={ndcg:.4f}, P={prec:.4f}, R={rec:.4f}")

=== Item-KNN GridSearch ===

Neighbors=5, TopK=50:
  @10: NDCG=0.4333, P=0.2082, R=0.0381
  @15: NDCG=0.4424, P=0.1933, R=0.0508
  @50: NDCG=0.4576, P=0.1450, R=0.1134

Neighbors=5, TopK=75:
  @10: NDCG=0.4333, P=0.2082, R=0.0381
  @15: NDCG=0.4424, P=0.1933, R=0.0508
  @50: NDCG=0.4576, P=0.1450, R=0.1134

Neighbors=5, TopK=100:
  @10: NDCG=0.4333, P=0.2082, R=0.0381
  @15: NDCG=0.4424, P=0.1933, R=0.0508
  @50: NDCG=0.4576, P=0.1450, R=0.1134

Neighbors=10, TopK=50:
  @10: NDCG=0.4247, P=0.2096, R=0.0383
  @15: NDCG=0.4351, P=0.1953, R=0.0519
  @50: NDCG=0.4556, P=0.1512, R=0.1198

Neighbors=10, TopK=75:
  @10: NDCG=0.4247, P=0.2096, R=0.0383
  @15: NDCG=0.4351, P=0.1953, R=0.0519
  @50: NDCG=0.4556, P=0.1512, R=0.1198

Neighbors=10, TopK=100:
  @10: NDCG=0.4247, P=0.2096, R=0.0383
  @15: NDCG=0.4351, P=0.1953, R=0.0519
  @50: NDCG=0.4556, P=0.1512, R=0.1198

Neighbors=20, TopK=50:
  @10: NDCG=0.4168, P=0.2049, R=0.0380
  @15: NDCG=0.4282, P=0.1932, R=0.0518
  @50: NDCG=0.4509, P=0.1

In [12]:
user_sim = cosine_similarity(R_binary, dense_output=False)

from scipy.sparse import lil_matrix

def user_knn_recommendations_fast(R_train, user_sim, top_k=50, top_neighbors=10, batch_size=500):
    n_users, n_items = R_train.shape
    recommendations = {}
    
    # 1. Обрезаем матрицу схожестей через LIL
    user_sim_lil = lil_matrix(user_sim.shape)
    for i in range(n_users):
        row = user_sim[i].toarray().flatten()
        row[i] = -np.inf
        top_indices = np.argsort(row)[::-1][:top_neighbors]
        user_sim_lil[i, top_indices] = user_sim[i, top_indices]
    
    user_sim_top = user_sim_lil.tocsr()  # → CSR для быстрого умножения
    
    # 2. Матричные операции
    for start in range(0, n_users, batch_size):
        end = min(start + batch_size, n_users)
        
        sim_batch = user_sim_top[start:end, :]
        scores_batch = sim_batch.dot(R_train)
        scores_batch = scores_batch.toarray() if hasattr(scores_batch, 'toarray') else scores_batch
        
        rated = R_train[start:end, :].toarray() > 0
        scores_batch[rated] = -np.inf
        
        top_indices = np.argsort(scores_batch, axis=1)[:, ::-1][:, :top_k]
        
        for i, recs in enumerate(top_indices):
            recommendations[start + i] = recs.tolist()
    
    return recommendations

# 2. Генерация рекомендаций
userknn_recs = user_knn_recommendations_fast(
    R_binary, 
    user_sim, 
    top_k=50, 
    top_neighbors=10, 
    batch_size=500
)

# 3. Оценка
ndcg, prec, rec = evaluate_recommendations(test_df, userknn_recs, k_ndcg=15, k_prec=15, k_rec=50)
print('User-KNN - NDCG@15: {:.4f}, P@15: {:.4f}, R@50: {:.4f}'.format(ndcg, prec, rec))

User-KNN - NDCG@15: 0.4109, P@15: 0.1857, R@50: 0.1257


In [16]:
print("=== User-KNN GridSearch ===")

# Параметры для перебора
neighbors_list = [5, 10, 20, 50]  # сколько соседей учитывать
top_k_list = [50, 75, 100]        # сколько рекомендовать

# Предвычисляем user_sim один раз
user_sim = cosine_similarity(R_binary, dense_output=False)

for neighbors in neighbors_list:
    for top_k in top_k_list:
        # Генерация рекомендаций
        userknn_recs = user_knn_recommendations_fast(
            R_binary, 
            user_sim, 
            top_k=top_k, 
            top_neighbors=neighbors, 
            batch_size=500
        )
        
        # Оценка по разным K
        print(f"\nNeighbors={neighbors}, TopK={top_k}:")
        
        # @10, @15, @50
        for k_ndcg, k_prec, k_rec in [(10,10,10), (15,15,15), (50,50,50)]:
            ndcg, prec, rec = evaluate_recommendations(
                test_df, userknn_recs,
                k_ndcg=k_ndcg, 
                k_prec=k_prec, 
                k_rec=k_rec
            )
            print(f"  @{k_ndcg}: NDCG={ndcg:.4f}, P={prec:.4f}, R={rec:.4f}")

=== User-KNN GridSearch ===

Neighbors=5, TopK=50:
  @10: NDCG=0.3502, P=0.1632, R=0.0309
  @15: NDCG=0.3681, P=0.1592, R=0.0445
  @50: NDCG=0.4069, P=0.1345, R=0.1119

Neighbors=5, TopK=75:
  @10: NDCG=0.3502, P=0.1632, R=0.0309
  @15: NDCG=0.3681, P=0.1592, R=0.0445
  @50: NDCG=0.4069, P=0.1345, R=0.1119

Neighbors=5, TopK=100:
  @10: NDCG=0.3502, P=0.1632, R=0.0309
  @15: NDCG=0.3681, P=0.1592, R=0.0445
  @50: NDCG=0.4069, P=0.1345, R=0.1119

Neighbors=10, TopK=50:
  @10: NDCG=0.3970, P=0.1940, R=0.0375
  @15: NDCG=0.4109, P=0.1857, R=0.0522
  @50: NDCG=0.4394, P=0.1503, R=0.1257

Neighbors=10, TopK=75:
  @10: NDCG=0.3970, P=0.1940, R=0.0375
  @15: NDCG=0.4109, P=0.1857, R=0.0522
  @50: NDCG=0.4394, P=0.1503, R=0.1257

Neighbors=10, TopK=100:
  @10: NDCG=0.3970, P=0.1940, R=0.0375
  @15: NDCG=0.4109, P=0.1857, R=0.0522
  @50: NDCG=0.4394, P=0.1503, R=0.1257

Neighbors=20, TopK=50:
  @10: NDCG=0.4275, P=0.2152, R=0.0416
  @15: NDCG=0.4385, P=0.2031, R=0.0564
  @50: NDCG=0.4597, P=0.1

## PureSVD

In [13]:
# задаём количество латентных факторов (гиперпараметр)
n_factors = 50

U, S, Vt = svds(R_ratings, k=n_factors)
S = np.diag(S)  # делаем диагональную матрицу

##### Что получаем ? ######
# U (n_users x n_factors) - профили пользователей в латентном пространстве (каждая строка = один пользователь; каждый столбец = один скрытый фактор (тема))
# S (n_factors x n_facrots) - важность каждого фактора
# Vt (n_factors x n_items) - профили фильмов в латентном пространстве (каждый столбец = фильм; каждая строка = один скрытый фактор)
############################

# Пример для пользователя 42:
##################################################      U[42] = [0.9, 0.1, 0.3, ...]  # 50 чисел
# Интерпретация:
# Фактор 1 (экшн): 0.9 → очень любит экшн
# Фактор 2 (романтика): 0.1 → не любит романтику
# Фактор 3 (комедия): 0.3 → немного любит комедии


# прогнозные рейтинги: full dense (для небольших матриц) или топ-K
R_pred = U @ S @ Vt  # восстановление матрицы

# Вместо полной R_pred вычисляем рекомендации на лету
def puresvd_recommendations_safe(U, S, Vt, R_train, top_k=50, batch_size=1000):
    n_users, n_items = R_train.shape
    recommendations = {}
    
    # Предвычисляем: V_scaled = S @ Vt (маленькая матрица) - оптимизация вычислений
    V_scaled = S @ Vt  # (n_factors × n_items)
    
    for start in range(0, n_users, batch_size): 
        end = min(start + batch_size, n_users)
        U_batch = U[start:end, :]  # (batch × n_factors)
        
        # Вычисляем предсказания только для батча
        pred_batch = U_batch @ V_scaled  # (batch × n_items)
        # генерация рекомендация    
        for i in range(pred_batch.shape[0]):
            scores = pred_batch[i, :].copy() # пердсказания одного пользователя
            user_id = start + i
            rated_indices = R_train[user_id].nonzero()[1] # что пользователь уже смотрел?
            scores[rated_indices] = -np.inf
            recs = np.argsort(scores)[::-1][:top_k]
            recommendations[user_id] = recs.tolist()
    
    return recommendations

# Использование:
svd_recs = puresvd_recommendations_safe(U, S, Vt, R_ratings, top_k=50)

# оценка
ndcg, prec, rec = evaluate_recommendations(test_df, svd_recs, k_ndcg=15, k_prec=15, k_rec=50)
print('PureSVD - NDCG@15: {:.4f}, P@15: {:.4f}, R@50: {:.4f}'.format(ndcg, prec, rec))

PureSVD - NDCG@15: 0.5014, P@15: 0.2378, R@50: 0.1568


In [17]:
print("=== PureSVD GridSearch ===")

# Параметры
n_factors_list = [20, 50, 100]  # сколько скрытых факторов
top_k_list = [50, 100]           # сколько рекомендовать

for n_factors in n_factors_list:
    print(f"\n=== n_factors={n_factors} ===")
    
    # Обучаем SVD
    U, S, Vt = svds(R_ratings, k=n_factors)
    S = np.diag(S)
    
    for top_k in top_k_list:
        # Генерация рекомендаций
        svd_recs = puresvd_recommendations_safe(
            U, S, Vt, 
            R_ratings, 
            top_k=top_k,
            batch_size=1000
        )
        
        # Оценка по разным K
        print(f"\n  TopK={top_k}:")
        
        for eval_k in [10, 15, 50]:
            ndcg, prec, rec = evaluate_recommendations(
                test_df, svd_recs,
                k_ndcg=eval_k,
                k_prec=eval_k,
                k_rec=eval_k
            )
            print(f"    @{eval_k:2d}: NDCG={ndcg:.4f}, P={prec:.4f}, R={rec:.4f}")

=== PureSVD GridSearch ===

=== n_factors=20 ===

  TopK=50:
    @10: NDCG=0.4798, P=0.2485, R=0.0485
    @15: NDCG=0.4881, P=0.2318, R=0.0657
    @50: NDCG=0.5010, P=0.1788, R=0.1525

  TopK=100:
    @10: NDCG=0.4798, P=0.2485, R=0.0485
    @15: NDCG=0.4881, P=0.2318, R=0.0657
    @50: NDCG=0.5010, P=0.1788, R=0.1525

=== n_factors=50 ===

  TopK=50:
    @10: NDCG=0.4944, P=0.2548, R=0.0513
    @15: NDCG=0.5014, P=0.2378, R=0.0687
    @50: NDCG=0.5127, P=0.1824, R=0.1568

  TopK=100:
    @10: NDCG=0.4944, P=0.2548, R=0.0513
    @15: NDCG=0.5014, P=0.2378, R=0.0687
    @50: NDCG=0.5127, P=0.1824, R=0.1568

=== n_factors=100 ===

  TopK=50:
    @10: NDCG=0.4910, P=0.2521, R=0.0497
    @15: NDCG=0.4995, P=0.2336, R=0.0664
    @50: NDCG=0.5104, P=0.1793, R=0.1526

  TopK=100:
    @10: NDCG=0.4910, P=0.2521, R=0.0497
    @15: NDCG=0.4995, P=0.2336, R=0.0664
    @50: NDCG=0.5104, P=0.1793, R=0.1526


# **Выводы:**

В ходе работы были построены и сравнены несколько классических алгоритмов рекомендательных систем на данных MovieLens. Основная задача - понять, какие классические подходы лучше работают на исторических данных оценок пользователей для дальнейшего добавления признаков и улучшения базовых показетелей качества моделей, построения более сложных моделей (ALS, TimeSVD++, NCF и т.д.)

### Подготовка данных

Данные содержали 32 миллиона оценок от 200 тысяч пользователей по 84 тысячам фильмов. Данные были разделены по времени: все, что до 1 января 2018 года, пошло в обучение, а оценки после этой даты = в тест. Это имитирует реальную ситуацию, когда мы обучаемся на прошлом и предсказываем будущее.

После разделения были оставлены только те пользователи и фильмы, которые присутствуют в обоих наборах - так мы оцениваем модель только на тех, кого она "знает".

### Использованные модели

Изначально были построены простые бейзлайны, чтобы понять минимальный уровень качества:

1. Случайные рекомендации - просто показываем случайные фильмы
2. Самые популярные - рекомендуем всем одни и те же топ-фильмы

Затем перешли к более сложным моделям:

1. Item-KNN - если вам понравился фильм X, то понравятся похожие на него фильмы
2. User-KNN - если вам нравится то же, что и другим похожим пользователям, то вам понравятся их другие выборы
3. PureSVD - раскладываем матрицу оценок на скрытые факторы (темы)

### Результаты

Случайные рекомендации показали ожидаемо низкое качество - около 0.1-0.5% точности. Это наш "нулевой" уровень.

Самые популярные фильмы дали уже значимое улучшение - @50: NDCG=0.2713, P=0.0772, R=0.0717.
Интересно, что просто показывать хиты всем - уже неплохая стратегия.

KNN-подходы (и Item-, и User-) показали значительное улучшение:

1.  ItemKNN для Neighbors=5, TopK=50, @50: NDCG=0.4576, P=0.1450, R=0.1134

2.  UserKNN для Neighbors=50, TopK=50, @50: NDCG=0.4705, P=0.1647, R=0.1377

PureSVD стал лидером - матричная факторизация лучше всего уловила сложные паттерны во вкусах пользователей. Для  n_factors=50, @50: NDCG=0.5127, P=0.1824, R=0.1568

### Что можно улучшить

Хотя текущие результаты хороши, есть потенциал для роста. Основное ограничение - мы использовали только факт "кто что оценил", игнорируя другую информацию:

1. Временные паттерны.
2. Контент фильмов - жанры, актеры, режиссеры
3. Характеристики и предпочтения пользователей

Проблема в том, что текущие модели (KNN, SVD) не умеют работать с такой дополнительной информацией. Для этого нужны гибридные подходы, которые объединяют коллаборативную фильтрацию с контентными признаками.

Кроме того, можно попытаться поробовать построить более сложные модели (ALS, NCF), которые могут показать более качественные результаты.

Таким образом, несмотря на относительную несложность моделей. даже такие простые алгоритмы коллаборативной фильтрации дают значимое улучшение по сравнению с базовыми подходами. PureSVD показал себя лучше всего на наших данных. Дальнейшее улучшение рекомендаций потребует использования дополнительной информации о пользователях и контенте через более сложные гибридные модели.